## **Summary**
This notebook demonstrates how to predict transcription factor perturbation effects without retraining a model, by using prior knowledge of transcription factor (TF)-mRNA interactions. The workflow includes:
1. Loading prior TF–target interaction data.
2. Loading and preprocessing RNA-seq dataset.
3. Predicting effects of TF perturbation.
4. Comparing predictions with actual experimental results using correlation.

## **Importing required packages**

In [1]:
#load packages
import decoupler as dc
import scanpy as sc
import tfactprofiler as tfp
import pandas as pd
from scipy import stats

/opt/anaconda3/envs/tfactprofiler5/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Loading prior knowledge (transcription factor–target mRNA interactions)**
This repository includes curated priors that encode transcription factor–to–mRNA relationships. The file, TFactprofiler_all.csv, aggregates interactions learned from a diverse collection of organs and cell types. This makes it the recommended default when broad generalizability across tissues is desired. For studies that require tissue- or cell-type-specificity, the tfact_organ directory provides models that were trained separately using data from each organ or cell type. You can select the file named after the organ of interest to bias the inference toward that context. All files consist of a transcription factor (source), target, and an interaction weight. One practical approach is to start with the TFactprofiler_all.csv file to establish a baseline. Then, if needed, repeat the analysis with the organ-specific file to assess tissue-specific effects or prioritize regulatory edges enriched in the focal context.

In [2]:
tfactprofiler_data=pd.read_csv("TFactprofiler_all.csv")
tfactprofiler_data

,source,target,weight
0,ADNP,BECN1,0.015420
1,ADNP2,HBB,0.002102
2,AEBP1,TFAP2A,-0.016116
3,AFP,ACAP1,0.013770
4,AFP,ACTBL2,0.032874
...,...,...,...
2606171,ZSCAN10,ZSCAN10,0.379152
2606172,SIX2,SIX2,0.379152
2606173,MYOCD,MYOCD,0.379152
2606174,SIRT1,SIRT1,0.379152


## **Loading RNA-seq data**.   
Here we use KnockTF dataset as an example.  

In [3]:
adata = dc.ds.knocktf()
KO_data=pd.DataFrame(adata.X,index=adata.obs["source"].values, columns=adata.var.reset_index()["index"].values)
KO_data=KO_data.loc[KO_data.index.intersection(tfactprofiler_data['source'].unique())]
KO_data=KO_data.loc[KO_data.index.intersection(tfactprofiler_data['target'].unique())]
KO_data = KO_data[~KO_data.index.duplicated(keep='first')]
KO_data=KO_data.T.reset_index()
KO_data.rename(columns={'index': 'Gene_Symbol'}, inplace=True)
KO_data=KO_data.set_index('Gene_Symbol')
KO_data

,PROX1,NR2F2,WT1,TP53,FLI1,ERG,YY1,STAT1,MITF,PAX3,...,TWIST2,ZHX2,ZNF750,CDC5L,CHCHD3,FUBP1,FUBP3,NKRF,SMAD4,SRPK2
Gene_Symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,-0.41056,-0.50900,-0.25537,0.05667,0.00000,-0.68861,-0.60616,0.6950,-0.58464,0.19841,...,-0.079202,-0.040975,0.196046,-0.869266,-0.267734,1.178337,-0.088848,1.345775,-0.457332,0.668721
A1BG-AS1,3.13675,2.12951,0.06701,0.00000,0.00000,0.41504,0.70886,0.0000,-0.16721,0.26759,...,0.000000,0.194022,-0.079479,-0.396422,-0.566347,0.652867,-0.514573,-0.501108,0.049598,0.000000
A1CF,-0.49427,-0.16245,0.31111,0.81121,0.40251,-0.27109,0.53354,0.2625,-0.17115,0.19540,...,-0.103422,-0.106180,-0.239093,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.120893
A2LD1,0.00000,0.00000,0.00000,0.46618,0.00000,0.00000,0.00000,0.0000,0.00000,0.00000,...,-0.131863,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A2M,0.23968,0.02671,-0.02576,-0.45087,1.93429,-0.62484,0.14472,0.3100,0.81078,0.07643,...,-0.051352,-0.186155,-0.052976,1.157541,-2.700440,0.000000,-1.169925,0.000000,-2.047306,0.263564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11B,-0.05072,-0.22164,0.34284,0.32373,0.00000,-0.16453,0.03332,-1.1525,0.04068,-0.08305,...,-0.189420,0.007123,-0.263574,-0.101012,-0.037203,-0.080819,0.053018,-1.278506,-0.077612,0.014075
ZYX,0.17918,-0.13836,-0.09468,-0.30996,3.02897,0.64812,-0.04361,0.1125,-0.49052,-0.14916,...,-0.324624,0.225050,-0.001476,0.326304,-0.015269,0.489546,-0.255257,2.323374,-0.075226,-0.215169
ZZEF1,-0.17708,0.23519,0.16314,0.39868,0.33722,-0.60526,0.24779,-0.0575,0.15993,-0.09437,...,-0.146047,0.000000,-0.281508,-0.312614,-0.201858,0.152832,-0.038431,-0.064335,0.397363,0.141374


## **Predicting TF perturbation effects**. 
Here we use TP53 as an example TF.  
You can change the TF by modifying the Perturb_TF variable.

In [4]:
Perturb_TF="TP53"
perturb_result = tfp.perturbation_predict(pd.DataFrame(KO_data[Perturb_TF]),
                                            tfactprofiler_data,perturb_tf=Perturb_TF,perturb_value=0)
perturb_result

,Gene_Symbol,Predicted_TP53
0,A1BG,0.064442
1,A1CF,0.765254
2,A2M,-0.431878
3,A2ML1,0.298377
4,A4GALT,0.007451
...,...,...
16145,ZYG11A,1.820106
16146,ZYG11B,0.309877
16147,ZYX,-0.337809
16148,ZZEF1,0.331778


## **Comparing predicted vs. actual perturbation effects**

In [6]:
KO_data_sub=KO_data.reset_index()[["Gene_Symbol",Perturb_TF]]
perturb_result_sub=perturb_result[["Gene_Symbol","Predicted_"+str(Perturb_TF)]]
sub=pd.merge(KO_data_sub,perturb_result_sub,on="Gene_Symbol",how="inner")
sub["Dif"]=sub["Predicted_"+str(Perturb_TF)]-sub[Perturb_TF]
sub1=sub[sub["Dif"]!=0]
r_p,p=stats.pearsonr(sub[Perturb_TF],-sub["Dif"]) 
print("Pearson correlation coefficient:", r_p)
print("p-value:", p)

Pearson correlation coefficient: 0.26379573159396275
p-value: 2.8836056927894734e-255
